## 1. Importation bibliothèques et nettoyage données

In [ ]:
!pip install catboost
!pip install category_encoders
!pip install --upgrade scikit-learn xgboost
!pip install h3
!pip install griddata
!pip install imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import matplotlib.dates as mdates
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import RandomForestClassifier as rf
from catboost import CatBoostRegressor, CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, roc_auc_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.svm import SVR
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
import scipy.interpolate as spi
from scipy.interpolate import griddata
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
# incendie hérault
data_incendie = pd.read_csv('C:/Users/wittl/OneDrive/Documents/Cours/Université et Etudes/IUT Belfort/S6/Stage/data/incendies_34.csv')
# meteo hérault
data_meteo = pd.read_csv('C:/Users/wittl/OneDrive/Documents/Cours/Université et Etudes/IUT Belfort/S6/Stage/data/departement-34-herault/data/meteostat/meteostat.csv')

In [ ]:
import os

output_path = "../results_img_csv"
os.makedirs(output_path, exist_ok=True)

In [ ]:
data_incendie.drop_duplicates()
data_meteo.drop_duplicates()

data_incendie.dropna()
data_meteo.dropna()

data_meteo.select_dtypes(include=['float64', 'int64']).columns
data_incendie.select_dtypes(include=['float64', 'int64']).columns

---
## 2. Visualisation graphique

### 2.1 Visualisation nombre d'incendies

In [ ]:
data_incendie['Date de première alerte'] = pd.to_datetime(data_incendie['Date de première alerte'])
fires_per_day = data_incendie.groupby(data_incendie['Date de première alerte'].dt.date).size()

plt.figure(figsize=(12, 6))
plt.plot(fires_per_day.index, fires_per_day.values)
plt.xlabel('Date')
plt.ylabel('Nombre incendies')
plt.tight_layout()
plt.show()

In [ ]:
data_incendie['Surface parcourue (m2)'].describe()

---
### 2.2 Visualisation données météo

In [ ]:
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'])
data_meteo['annee'] = data_meteo['creneau'].dt.year

plt.plot(data_meteo['creneau'], data_meteo['temp'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Température (°C)")
plt.show()

In [ ]:
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'])
data_meteo['annee'] = data_meteo['creneau'].dt.year

plt.plot(data_meteo['creneau'], data_meteo['ffmc'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Indice FFMC")
plt.show()

In [ ]:
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'])
data_meteo['annee'] = data_meteo['creneau'].dt.year

plt.plot(data_meteo['creneau'], data_meteo['dmc'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Indice DMC")
plt.show()

In [ ]:
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'])
data_meteo['annee'] = data_meteo['creneau'].dt.year

plt.plot(data_meteo['creneau'], data_meteo['isi'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Indice ISI")
plt.show()

In [ ]:
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'])
data_meteo['annee'] = data_meteo['creneau'].dt.year

plt.plot(data_meteo['creneau'], data_meteo['bui'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Indice BUI")
plt.show()

In [ ]:
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'])
data_meteo['annee'] = data_meteo['creneau'].dt.year

plt.plot(data_meteo['creneau'], data_meteo['fwi'], label='Température')
plt.xlabel("Créneau horaire")
plt.ylabel("Indice FWI")
plt.show()

---
### 2.3 Visualisation nb incendies sur carte

In [ ]:
LAT_MIN, LAT_MAX = 43.2, 44.0
LON_MIN, LON_MAX = 2.5, 4.2

filtered_data = data_incendie[
    (data_incendie['latitude'] >= LAT_MIN) &
    (data_incendie['latitude'] <= LAT_MAX) &
    (data_incendie['longitude'] >= LON_MIN) &
    (data_incendie['longitude'] <= LON_MAX)
]

if not filtered_data.empty:
    gridsize = 50
    x = filtered_data['longitude']
    y = filtered_data['latitude']

    counts, xedges, yedges = np.histogram2d(x, y, bins=gridsize)
    x_centers = (xedges[:-1] + xedges[1:]) / 2
    y_centers = (yedges[:-1] + yedges[1:]) / 2

    hex_centers = []
    for i in range(len(x_centers)):
        for j in range(len(y_centers)):
            if counts[i, j] > 0:
                hex_centers.append({
                    "lat": y_centers[j],
                    "lon": x_centers[i],
                    "count": counts[i, j]
                })

    map_center = [(LAT_MIN + LAT_MAX) / 2, (LON_MIN + LON_MAX) / 2]
    m = folium.Map(location=map_center, zoom_start=10)

    for hexagon in hex_centers:
        folium.CircleMarker(
            location=[hexagon["lat"], hexagon["lon"]],
            radius=5 + np.sqrt(hexagon["count"]),
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.6,
            tooltip=f"Incendies : {int(hexagon['count'])}"
        ).add_to(m)

    m.save('../results_img_csv/incendies_aggreges_herault_map.html')
    print("Carte enregistrée sous 'incendies_aggreges_herault_map.html'.")
else:
    print("Aucune donnée valide après nettoyage.")

---
## 3. Analyse exploratoire

---
### 3.1 Statistiques datasets

In [ ]:
data_incendie.describe()

In [ ]:
data_meteo.describe()

---
### 3.2 Distribution datasets (histogrammes/boxplots)

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(data_meteo['temp'], bins=50, color='purple', alpha=0.7)
plt.xlabel('Température')
plt.ylabel('Fréquence')
plt.show()

data_meteo = data_meteo.drop_duplicates()
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'], errors='coerce')
data_meteo = data_meteo.dropna(subset=['creneau', 'temp'])
data_meteo['Année'] = data_meteo['creneau'].dt.year.astype(str)

plt.figure(figsize=(12, 6))
sns.boxplot(data=data_meteo, x='Année', y='temp', palette='coolwarm')
plt.xlabel('Année')
plt.ylabel('Température (°C)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(data_meteo['fwi'], bins=50, color='purple', alpha=0.7)
plt.xlabel('Indice FWI')
plt.ylabel('Fréquence')
plt.show()

data_meteo = data_meteo.drop_duplicates()
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'], errors='coerce')
data_meteo = data_meteo.dropna(subset=['creneau', 'fwi'])
data_meteo['Année'] = data_meteo['creneau'].dt.year.astype(str)

plt.figure(figsize=(12, 6))
sns.boxplot(data=data_meteo, x='Année', y='fwi', palette='coolwarm')
plt.xlabel('Année')
plt.ylabel('Indice FWI')
plt.tight_layout()
plt.show()

---
### 3.3 Corrélations

In [ ]:
data_meteo['Mois'] = data_meteo['creneau'].dt.month
heatmap_meteo = data_meteo.groupby(['Année', 'Mois'])['temp'].mean().unstack()

plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_meteo, annot=True, fmt=".1f", cmap="inferno", cbar_kws={'label': 'Température (°C)'})
plt.xlabel('Mois')
plt.ylabel('Année')
plt.show()

In [ ]:
data_meteo['Mois'] = data_meteo['creneau'].dt.month
heatmap_meteo = data_meteo.groupby(['Année', 'Mois'])['fwi'].mean().unstack()

plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_meteo, annot=True, fmt=".1f", cmap="cividis", cbar_kws={'label': 'Indice FWI'})
plt.xlabel('Mois')
plt.ylabel('Année')
plt.show()

---
## 4. Modélisation

---
### 4.1 Création dataset + colonne cible

In [ ]:
data_meteo['creneau'] = pd.to_datetime(data_meteo['creneau'])
data_incendie['Date de première alerte'] = pd.to_datetime(data_incendie['Date de première alerte'])

data_incendie['Date de première alerte'] = data_incendie['Date de première alerte'].dt.strftime('%Y-%m-%d')

In [ ]:
temp_columns = ['temp']
average_temps = data_meteo.groupby('creneau')['temp'].mean()
print(average_temps)

In [ ]:
data_incendie['Date de première alerte'] = pd.to_datetime(data_incendie['Date de première alerte'])

data_incendie['incendie_present'] = 1

dates_incendie = data_incendie[['Date de première alerte']].drop_duplicates()
dates_incendie['incendie_present'] = 1

dates_meteo = pd.DataFrame(data_meteo['creneau'].dt.date.unique(), columns=['creneau'])

average_temps = data_meteo.groupby('creneau')['temp'].mean().reset_index()

merged_data = pd.merge(average_temps, dates_incendie, how='left', left_on='creneau', right_on='Date de première alerte')

merged_data['incendie_present'] = merged_data['incendie_present'].fillna(0)

merged_data = merged_data.rename(columns={'temp': 'moyenne_temp', 'creneau': 'date'})

merged_data = merged_data.drop(columns=['Date de première alerte'])

merged_data.describe()

merged_data.to_csv('../results_img_csv/merged_data34.csv', index=False)

In [ ]:
data_meteo['target'] = (data_meteo['fwi'] > data_meteo['fwi'].median()).astype(int)

---
### 4.2 Test scalers

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder


numeric_features = data_meteo.select_dtypes(include=['number']).columns
data_numeric = data_meteo[numeric_features]

standardscaler = StandardScaler()
scaled_data = standardscaler.fit_transform(data_numeric)

data_meteo[numeric_features] = scaled_data
print("Standard scaler : \n", scaled_data)

robustScaler = RobustScaler()
scaled_data = robustScaler.fit_transform(data_numeric)

data_meteo[numeric_features] = scaled_data
print("Robust scaler : \n", scaled_data)

minMaxScaler = MinMaxScaler()
scaled_data = minMaxScaler.fit_transform(data_numeric)

data_meteo[numeric_features] = scaled_data
print("MinMax scaler : \n", scaled_data)

---
### 4.3 Preprocessing

In [ ]:
def handle_outliers(df):
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

    lower_bounds = {}
    upper_bounds = {}

    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        lower_bounds[col] = lower_bound
        upper_bounds[col] = upper_bound

    df_cleaned = df.copy()

    for col in numeric_cols:
        median_value = df[col].median()
        df_cleaned[col] = df_cleaned[col].apply(
            lambda x: median_value if (x < lower_bounds[col] or x > upper_bounds[col]) else x
        )

    return df_cleaned

data_incendie_cleaned = handle_outliers(data_incendie)
data_meteo_cleaned = handle_outliers(data_meteo)

In [ ]:
def prepare_dataset_meteo(df, target_column, columns_to_drop):
    print("\n--- Étape : Nettoyage du Dataset ---")

    df_clean = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
    print(f"Dimensions après suppression des colonnes inutiles : {df_clean.shape}")

    if target_column not in df_clean.columns:
        raise ValueError(f"La colonne cible '{target_column}' est introuvable après suppression des colonnes.")

    non_numeric_columns = df_clean.select_dtypes(exclude=['int64', 'float64']).columns
    if len(non_numeric_columns) > 0:
        print(f"Colonnes non numériques détectées : {list(non_numeric_columns)}")

        for col in non_numeric_columns:
            if col in ['Direction du vent']:
                print(f"Encodage de la colonne non numérique : {col}")
                df_clean[col] = df_clean[col].astype('category').cat.codes
            else:
                print(f"Colonne non numérique supprimée : {col}")
                df_clean = df_clean.drop(columns=[col], errors='ignore')

    X = df_clean.drop(columns=[target_column])
    y = df_clean[target_column]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Taille des ensembles - Entraînement : {X_train.shape}, Test : {X_test.shape}")

    imputer = SimpleImputer(strategy='median')
    X_train_imputed = imputer.fit_transform(X_train)
    X_test_imputed = imputer.transform(X_test)

    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)
    X_test_scaled = scaler.transform(X_test_imputed)

    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_train_scaled, y_train)
    print(f"Distribution des classes après SMOTE :\n{pd.Series(y_resampled).value_counts()}")

    return X_resampled, X_test_scaled, y_resampled, y_test

In [ ]:
def prepare_dataset_incendie(df, target_column, columns_to_drop):
    print("\n--- Étape : Nettoyage du Dataset ---")

    df_clean = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')
    print(f"Dimensions après suppression des colonnes inutiles : {df_clean.shape}")

    if target_column not in df_clean.columns:
        raise ValueError(f"La colonne cible '{target_column}' est introuvable après suppression des colonnes.")

    non_numeric_columns = df_clean.select_dtypes(exclude=['int64', 'float64']).columns
    print(f"Colonnes non numériques détectées : {list(non_numeric_columns)}")

    for col in non_numeric_columns:
        if col == target_column:
            continue
        if col in ['Direction du vent', 'Connaissance de la cause', 'Nature']:
            print(f"Encodage de la colonne non numérique : {col}")
            df_clean[col] = df_clean[col].astype('category').cat.codes
        else:
            print(f"Colonne non numérique supprimée : {col}")
            df_clean = df_clean.drop(columns=[col], errors='ignore')

    print(f"Nombre de valeurs nulles avant traitement :\n{df_clean.isnull().sum()}")
    df_clean = df_clean.fillna(df_clean.median(numeric_only=True))
    print(f"Dimensions après traitement des valeurs nulles : {df_clean.shape}")

    X = df_clean.drop(columns=[target_column])
    y = df_clean[target_column]

    print(f"Dimensions de X : {X.shape}, Dimensions de y : {y.shape}")

    if X.empty or y.empty:
        raise ValueError("Les données sont vides après nettoyage. Vérifiez votre dataset.")

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)
    print(f"Taille des ensembles - Entraînement : {X_train.shape}, Test : {X_test.shape}")

    return X_train, X_test, y_train, y_test

In [ ]:
columns_to_drop_meteo = [
    'index', 'latitude', 'longitude', 'year', 'annee'
]

In [ ]:
columns_to_drop_incendie = [
    'Unnamed: 0.1', 'Unnamed: 0', 'Numéro', 'Code INSEE', 'Nom de la commune',
    'Précision des surfaces', 'Précision de la donnée', 'latitude', 'longitude', 'Année'
]

In [ ]:
"""try:
    print("\n=== Dataset Météo ===")
    columns_to_drop_meteo = ['index', 'latitude', 'longitude', 'year', 'annee']
    data_meteo_cleaned = handle_outliers(data_meteo)
    X_train_meteo, X_test_meteo, y_train_meteo, y_test_meteo = prepare_dataset_meteo(data_meteo_cleaned, target_column='target', columns_to_drop=columns_to_drop_meteo)

except ValueError as e:
    print(f"Erreur pour Dataset Météo : {e}")
"""
"""
try:
    print("\n=== Dataset Incendie ===")
    X_train_incendie, X_test_incendie, y_train_incendie, y_test_incendie = prepare_dataset_incendie(
        data_incendie_cleaned, target_column='target', columns_to_drop=columns_to_drop_incendie
    )
except ValueError as e:
    print(f"Erreur pour Dataset Incendie : {e}")
"""

In [ ]:
train_data = merged_data[merged_data['date'] <= '2022-12-31']
test_data = merged_data[merged_data['date'] >= '2023-01-01']

features = ['moyenne_temp']
target = 'incendie_present'

X_train = train_data[features]
y_train = train_data[target]

X_test = test_data[features]
y_test = test_data[target]

imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print(f"Données d'entraînement : {X_train_scaled.shape[0]} lignes")
print(f"Données de test : {X_test_scaled.shape[0]} lignes")

---
### 4.4 Scores des modèles

In [ ]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, max_depth=5, random_state=42),
    'CatBoost': CatBoostClassifier(learning_rate=0.1, depth=5, iterations=100, verbose=0, random_state=42),
    'LightGBM': LGBMClassifier(max_depth=5, n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', max_depth=5, n_estimators=100, learning_rate=0.1)
}
results = {}

In [ ]:
for model_name, model in models.items():
    try:
        cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='accuracy')

        results[model_name] = {
            'scores': cv_scores,
            'mean_score': np.mean(cv_scores),
            'std_dev': np.std(cv_scores)
        }
    except Exception as e:
        print(f"Erreur avec le modèle {model_name}: {e}")
        results[model_name] = {
            'scores': None,
            'mean_score': None,
            'std_dev': None
        }

for model_name, result in results.items():
    print(f"Modèle: {model_name}")
    if result['scores'] is not None:
        print(f"Scores de cross-validation: {result['scores']}")
        print(f"Moyenne des scores: {result['mean_score']}")
        print(f"Écart-type des scores: {result['std_dev']}")
    else:
        print("Erreur avec ce modèle")
    print('-' * 50)

---
Distribution classes

In [ ]:
print("Distribution des classes dans y_train de incendie :")
print(y_train.value_counts(normalize=True))
print("Distribution des classes dans y_test de incendie :")
print(y_test.value_counts(normalize=True))

---
### 4.5 Scores des métriques

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    print(f"\n=== Résultats pour {name} ===")
    y_pred = model.predict(X_test)
    y_proba = None
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print(f"Accuracy : {accuracy:.6f}")
    print(f"Precision : {precision:.6f}")
    print(f"Recall : {recall:.6f}")
    print(f"F1-Score : {f1:.6f}")

    cm = confusion_matrix(y_test, y_pred)
    print("\nMatrice de confusion :")
    print(cm)

    return accuracy, precision, recall, f1

In [ ]:
for name, model in models.items():
    print(f"\nEntraînement du modèle : {name}")
    model.fit(X_train_scaled, y_train)
    #evaluate_model(name, model, X_train_meteo, y_train_meteo, X_test_meteo, y_test_meteo) # data_meteo
    #evaluate_model(name, model, X_train_incendie, y_train_incendie, X_test_incendie, y_test_incendie) # data_incendie
    evaluate_model(name, model, X_train_scaled, y_train, X_test_scaled, y_test) # dataset_final

In [ ]:
# Pour Cross Validation
from sklearn.model_selection import cross_validate

scoring_metrics = ['accuracy', 'precision', 'recall', 'f1']

cv_results = {}

for model_name, model in models.items():
    try:
        scores = cross_validate(model, X_train_scaled, y_train, cv=5, scoring=scoring_metrics)
        cv_results[model_name] = {
            'accuracy': np.mean(scores['test_accuracy']),
            'precision': np.mean(scores['test_precision']),
            'recall': np.mean(scores['test_recall']),
            'f1-score': np.mean(scores['test_f1']),
            'accuracy_std': np.std(scores['test_accuracy']),
            'precision_std': np.std(scores['test_precision']),
            'recall_std': np.std(scores['test_recall']),
            'f1-score_std': np.std(scores['test_f1']),
        }

    except Exception as e:
        print(f"Erreur avec le modèle {model_name}: {e}")
        cv_results[model_name] = None

for model_name, result in cv_results.items():
    print(f"\n🔹 Modèle: {model_name}")
    if result:
        print(f"Accuracy: {result['accuracy']:.4f} ± {result['accuracy_std']:.4f}")
        print(f"Precision: {result['precision']:.4f} ± {result['precision_std']:.4f}")
        print(f"Recall: {result['recall']:.4f} ± {result['recall_std']:.4f}")
        print(f"F1-Score: {result['f1-score']:.4f} ± {result['f1-score_std']:.4f}")
    else:
        print("⚠️ Erreur avec ce modèle")
    print('-' * 50)